# Chapter 22 · The network finds its own clues
**Deck 4 · Live demo.** Train the 2-2-1 network (sigmoid neurons, log loss, η = 1) from a fixed start, and open the box.

**You will:** write forward and backward for 9 numbers · train 3,000 epochs · prove the loss table · see the hidden clues it invented.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
At which epoch are all four students first on the right side of 0.5: 100, 500 or 1,000?

In [ ]:
my_prediction = 500

In [ ]:
# The body-clock students: x1 = exam time (0 morning, 1 evening), x2 = body clock (0 early bird, 1 night owl)
names = ['Kabir', 'Zara', 'Tara', 'Omar']
X = np.array([[0, 0], [1, 1], [1, 0], [0, 1]], dtype=float)
y = np.array([1, 1, 0, 0], dtype=float)          # pass when exam time matches body clock
for n, (a, b), r in zip(names, X, y):
    print(f"{n:6} exam {'evening' if a else 'morning':8} {'night owl' if b else 'early bird':10} → {'pass' if r else 'fail'}")

## 2 · Fixed starting weights (same as the slides)

In [ ]:
sig = lambda z: 1 / (1 + np.exp(-z))
START = dict(W1=np.array([[0.5, -0.5], [0.4, -0.6]]), b1=np.array([0.1, 0.2]), W2=np.array([0.6, 0.7]), b2=0.0)

def forward(p):
    H = sig(X @ p['W1'] + p['b1'])
    P = sig(H @ p['W2'] + p['b2'])
    return H, P

def loss(p):
    P = forward(p)[1]
    return -np.mean(y * np.log(P) + (1 - y) * np.log(1 - P))

print('start P(pass):', forward(START)[1].round(2), '· loss', round(loss(START), 3))

## 3 · Backprop for all nine numbers
With sigmoid + log loss the output blame is simply (P − y) (Chapter 16). Then pass it back through W2 and each hidden sigmoid (slope h(1 − h)).

In [ ]:
def train(epochs, eta=1.0):
    p = {k: np.array(v, dtype=float) for k, v in START.items()}
    for _ in range(epochs):
        H, P = forward(p)
        d2 = (P - y) / len(y)                       # blame on the output
        gW2, gb2 = H.T @ d2, d2.sum()
        dH = np.outer(d2, p['W2']) * H * (1 - H)    # blame on each hidden neuron
        gW1, gb1 = X.T @ dH, dH.sum(axis=0)
        p['W1'] -= eta * gW1; p['b1'] -= eta * gb1; p['W2'] -= eta * gW2; p['b2'] -= eta * gb2
    return p

## 4 · The loss table

In [ ]:
table = {}
for e in (0, 100, 300, 500, 1000, 3000):
    p = train(e)
    table[e] = (round(loss(p), 3), forward(p)[1].round(2))
    print(f'epoch {e:>5}: loss {table[e][0]:.3f} · P(pass) {table[e][1]}')

## 5 · PROOF

In [ ]:
assert [table[e][0] for e in (0, 100, 300, 500, 1000, 3000)] == [0.748, 0.681, 0.472, 0.085, 0.017, 0.004]
first_all_right = next(e for e in (100, 300, 500, 1000) if np.all((table[e][1] >= 0.5) == (y == 1)))
assert first_all_right == my_prediction == 500
print('✓ proof: the slide table, all four right at epoch 500')

## 6 · Open the box: the hidden clues it invented

In [ ]:
final = train(3000)
H, P = forward(final)
for n, h, pr in zip(names, H.round(2), P.round(3)):
    print(f'{n:6} hidden {h} · P(pass) {pr}')
print('W1', final['W1'].round(2).tolist(), 'b1', final['b1'].round(2), 'W2', final['W2'].round(2), 'b2', float(final['b2']).__round__(2))
assert H[1, 0] > 0.9 and H[0, 1] > 0.9, 'hidden 1 = both evening (Zara), hidden 2 = both morning (Kabir)'

## 7 · Picture

In [ ]:
curve = [loss(train(e)) for e in range(0, 3001, 50)]
plt.plot(range(0, 3001, 50), curve); plt.xlabel('epoch'); plt.ylabel('log loss'); plt.title('plateau, then drop'); plt.show()

## 8 · Your turn
Try other starting weights. Does it always find the same two clues? Does it always succeed?

In [ ]:
# TODO

## Recap
Forward, blame back, step, repeat. The hidden neurons reinvented our hand-made clues, scaled up.